#  Data Cleansing & Standarisasi Dataset Penjualan (30 Data)

**Deskripsi**
Notebook ini membersihkan dataset penjualan

**Masalah data yang ditangani**

| Kolom | Masalah | Hasil standar |
|---|---|---|
| `Sale_Date` | Campuran format (`21/09/2023`, `04-22-2023`, `2023-04-22`, `03-Feb-2024`, `08 November 2023`) | `YYYY-MM-DD` |
| `Sales_Rep`, `Region`, `Sales_Channel`, `Customer_Type` | Huruf campur (`bob`, `BOB`, `north`, `NORTH`, `ONLINE`) | Title Case (`Bob`, `North`, `Online`) |
| `Product_Category` | Variasi ejaan (`Foods`, `FOOD`, `Cloths`, `clothing`, `Electronic Goods`, `Furnitures`) | `Food`, `Clothing`, `Electronics`, `Furniture` |
| `Payment_Method` | Variasi ejaan (`BANK TRANSFER`, `bank trans`, `Creditcard`, `cash`) | `Bank Transfer`, `Wire Transfer`, `Credit Card`, `Cash` |
| `Sales_Amount` | Ada simbol `$` dan pemisah ribuan (`$6,551.23`) | Angka desimal (`6551.23`) |
| `Discount` | Campuran `0.2`, `18%`, dan `9` | Pecahan 0–1 (`0.2`, `0.18`, `0.09`) |
| `Unit_Cost`, `Unit_Price` | Ada nilai kosong | Diisi median per kategori produk |
| Kolom turunan (`Rep_Name`, `Category_*`, `Region_*`) | Duplikasi isi kolom lain dengan format berantakan | Dihapus (bisa dimatikan) |
| Baris | Duplikat (mis. ID 1088, 1064, 1052) | Dihapus setelah standarisasi |

**Struktur folder di Drive**
```
MyDrive/Data_Cleansing/
 ├── raw/    → dataset asli (sebelum cleansing)
 └── clean/  → dataset bersih (sesudah cleansing)
```

## 1. Mount Google Drive & Pengaturan Folder

In [1]:
import os
import re
import warnings
import numpy as np
import pandas as pd
from google.colab import drive

warnings.filterwarnings('ignore')
drive.mount('/content/drive')

# ====== PENGATURAN PATH ======
BASE_DIR  = '/content/drive/MyDrive/Data_Cleansing'
RAW_DIR   = f'{BASE_DIR}/raw'     # dataset sebelum cleansing
CLEAN_DIR = f'{BASE_DIR}/clean'   # dataset sesudah cleansing

RAW_FILE   = 'dataset_penjualan_mentah.xlsx'   # ganti sesuai nama file (.xlsx / .csv)
CLEAN_FILE = 'dataset_penjualan_bersih.csv'

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)
print('Folder siap:', BASE_DIR)

Mounted at /content/drive
Folder siap: /content/drive/MyDrive/Data_Cleansing


## 2. Muat Dataset (Sebelum Cleansing)
Jika file belum ada di Drive, kamu akan diminta mengunggahnya. File asli otomatis disimpan ke folder `raw`.

In [2]:
raw_path = f'{RAW_DIR}/{RAW_FILE}'

if not os.path.exists(raw_path):
    from google.colab import files
    print('File belum ada di Drive. Silakan unggah dataset:')
    uploaded = files.upload()
    for name, content in uploaded.items():
        with open(f'{RAW_DIR}/{name}', 'wb') as f:
            f.write(content)
    RAW_FILE = list(uploaded.keys())[0]
    raw_path = f'{RAW_DIR}/{RAW_FILE}'

def baca_file(path):
    if path.lower().endswith(('.xlsx', '.xls')):
        return pd.read_excel(path)
    for enc in ('utf-8', 'utf-8-sig', 'latin-1'):
        try:
            return pd.read_csv(path, encoding=enc, sep=None, engine='python')
        except Exception:
            continue
    raise ValueError('Gagal membaca file.')

df_raw = baca_file(raw_path)
print(f'Dataset dimuat: {df_raw.shape[0]} baris x {df_raw.shape[1]} kolom')
display(df_raw)
print('\nTipe data, nilai kosong, dan jumlah nilai unik:')
display(pd.DataFrame({'dtype': df_raw.dtypes, 'kosong': df_raw.isna().sum(), 'unik': df_raw.nunique()}))

File belum ada di Drive. Silakan unggah dataset:


Saving sales_data_.xlsx to sales_data_.xlsx
Dataset dimuat: 34 baris x 16 kolom


,Product_ID,Sale_Date,Sales_Rep,Region,Sales_Amount,Quantity_Sold,Product_Category,Unit_Cost,Unit_Price,Customer_Type,Discount,Payment_Method,Sales_Channel,Region_and_Sales_Rep,Category_Copy,Rep_Name
0,1015,21/09/2023,David,SOUTH,4631.23,30.0,Foods,261.56,371.40,Returning,0.2,Bank Transfer,Retail,SOUTH David,FOODS,DAVID
1,1002,04-22-2023,Eve,north,"$6,551.23",9.0,Electronic,4398.16,4439.12,new,18%,Wire Transfer,Online,north-Eve,electronic,EVE
2,1093,2023-04-21,bob,WEST,"$4,384.02",17.0,FURNITURE,3816.39,4209.44,Returning,0.11,NaN,Retail,bob (WEST),FURNITURE,BOB
3,1075,09/10/2023,Charlie,WEST,8239.58,18.0,Cloths,2228.35,2682.34,New,13%,Wire Transfer,Online,WEST Charlie,CLOTHS,CHARLIE
4,1003,2023-12-31,Alice,SOUTH,4775.59,30.0,FURNITURE,4190.28,4270.65,new,20%,Cash,Online,SOUTH-Alice,furniture,ALICE
5,1021,02-11-2023,CHARLIE,west,3761.15,32.0,FOOD,900.79,NaN,new,0.21,CASH,online,CHARLIE (west),FOOD,CHARLIE
6,1064,2023-01-04,Eve,East,7412.11,10.0,Electronic Goods,4764.96,5074.42,New,12%,Cash,ONLINE,East-Eve,ELECTRONIC GOODS,EVE
7,1088,04/10/2023,BOB,EAST,6116.75,40.0,Electronic,4904.93,5034.35,New,0.1,Creditcard,retail,EAST-BOB,electronic,BOB
8,1088,04/10/2023,BOB,EAST,6116.75,40.0,Electronic,4904.93,5034.35,New,0.1,Creditcard,retail,EAST-BOB,electronic,BOB
9,1030,2023-07-17,David,WEST,"$3,023.48",19.0,clothing,3049.33,3209.22,Returning,26%,cash,online,WEST David,clothing,DAVID



Tipe data, nilai kosong, dan jumlah nilai unik:


,dtype,kosong,unik
Product_ID,int64,0,25
Sale_Date,object,0,30
Sales_Rep,object,0,16
Region,object,0,16
Sales_Amount,object,0,30
Quantity_Sold,float64,2,20
Product_Category,object,0,15
Unit_Cost,float64,2,28
Unit_Price,float64,3,28
Customer_Type,object,2,8


## 3. Konfigurasi

In [3]:
# Format angka pada kolom uang:
#   'us' -> 1,234.56  (koma = ribuan, titik = desimal)  <- sesuai data ($6,551.23)
#   'id' -> 1.234,56  (titik = ribuan, koma = desimal)
FORMAT_ANGKA = 'us'

HAPUS_KOLOM_REDUNDAN = True   # hapus Rep_Name, Category_*, Region_* (duplikasi kolom lain)
HAPUS_DUPLIKAT       = True
ISI_DISKON_KOSONG    = 0      # diskon kosong dianggap tidak ada diskon
ISI_TEKS_KOSONG      = 'Tidak Diketahui'

# Nilai yang dianggap kosong
NILAI_KOSONG = ['', '-', '--', 'n/a', 'na', 'null', 'none', 'nan', '?', 'tidak ada', 'kosong']

## 4. Cleansing Dasar (Nama Kolom, Spasi, Nilai Kosong)

In [4]:
log = []
df = df_raw.copy()

# 4.1 Nama kolom -> snake_case
def snake(s):
    s = str(s).strip().lower()
    s = re.sub(r'[^\w\s]', '', s)
    return re.sub(r'\s+', '_', s)

df.columns = [snake(c) for c in df.columns]
log.append('Nama kolom diseragamkan ke snake_case')

# 4.2 Cari kolom berdasarkan awalan nama (tahan terhadap perbedaan penamaan)
def kolom(awalan, hindari=()):
    if awalan in df.columns:
        return awalan
    kandidat = [c for c in df.columns
                if c.startswith(awalan) and not any(c.startswith(h) for h in hindari)]
    return kandidat[0] if kandidat else None

K = {
    'id':       kolom('product_i'),
    'tanggal':  kolom('sale_date'),
    'rep':      kolom('sales_rep'),
    'region':   kolom('region', hindari=('region_ar',)),
    'amount':   kolom('sales_am'),
    'qty':      kolom('quantity'),
    'kategori': kolom('product_c'),
    'cost':     kolom('unit_cost'),
    'price':    kolom('unit_price'),
    'cust':     kolom('customer'),
    'diskon':   kolom('discount'),
    'bayar':    kolom('payment'),
    'channel':  kolom('sales_cha'),
}
print('Pemetaan kolom:')
for k, v in K.items():
    print(f'  {k:9s} -> {v}')
tidak_ada = [k for k, v in K.items() if v is None]
if tidak_ada:
    print('\n⚠️ Kolom tidak ditemukan:', tidak_ada, '- periksa nama kolom di dataset.')

# 4.3 Hapus kolom redundan (turunan dari kolom lain)
if HAPUS_KOLOM_REDUNDAN:
    dipakai = set(v for v in K.values() if v)
    redundan = [c for c in df.columns
                if c.startswith(('rep_name', 'category_', 'region_ar')) and c not in dipakai]
    df = df.drop(columns=redundan)
    log.append(f'Kolom redundan dihapus: {redundan}')

# 4.4 Rapikan spasi & ubah placeholder kosong -> NaN
for c in df.select_dtypes(include=['object', 'string']).columns:
    df[c] = df[c].astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)
    df[c] = df[c].mask(df[c].str.lower().isin(NILAI_KOSONG))
log.append('Spasi berlebih dirapikan & placeholder kosong diubah menjadi NaN')

# 4.5 Hapus baris kosong total
n0 = len(df)
df = df.dropna(how='all')
log.append(f'Baris kosong total dihapus: {n0 - len(df)}')
print('\nCleansing dasar selesai.')

Pemetaan kolom:
  id        -> product_id
  tanggal   -> sale_date
  rep       -> sales_rep
  region    -> region
  amount    -> sales_amount
  qty       -> quantity_sold
  kategori  -> product_category
  cost      -> unit_cost
  price     -> unit_price
  cust      -> customer_type
  diskon    -> discount
  bayar     -> payment_method
  channel   -> sales_channel

Cleansing dasar selesai.


## 5. Standarisasi Kolom
Setiap kolom diseragamkan dengan aturan masing-masing. Catatan format tanggal:
- `21/09/2023` (garis miring) dibaca **DD/MM/YYYY**
- `04-22-2023` (garis strip, angka semua) dibaca **MM-DD-YYYY**
- `2023-04-22` dan `2023/10/15` dibaca **YYYY-MM-DD**
- `03-Feb-2024` dan `08 November 2023` dibaca apa adanya (nama bulan)

In [5]:
# ---------- Fungsi bantu ----------
def terap(key, fn):
    c = K.get(key)
    if c is not None and c in df.columns:
        df[c] = df[c].apply(fn)

def title(x):
    return str(x).title() if pd.notna(x) else x

def std_kategori(x):
    if pd.isna(x): return x
    s = str(x).lower()
    if 'food' in s:    return 'Food'
    if 'electr' in s:  return 'Electronics'
    if 'furnit' in s:  return 'Furniture'
    if 'cloth' in s:   return 'Clothing'
    return str(x).title()

def std_bayar(x):
    if pd.isna(x): return x
    s = str(x).lower()
    if 'bank' in s:    return 'Bank Transfer'
    if 'wire' in s:    return 'Wire Transfer'
    if 'credit' in s:  return 'Credit Card'
    if 'cash' in s:    return 'Cash'
    return str(x).title()

def std_cust(x):
    if pd.isna(x): return x
    s = str(x).lower()
    if 'new' in s:     return 'New'
    if 'return' in s:  return 'Returning'
    return str(x).title()

def ke_angka(x):
    if pd.isna(x): return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)): return float(x)
    s = re.sub(r'[^\d,.\-]', '', str(x))
    if s in ('', '-', '.', ','): return np.nan
    s = s.replace(',', '') if FORMAT_ANGKA == 'us' else s.replace('.', '').replace(',', '.')
    try:
        return float(s)
    except ValueError:
        return np.nan

def std_diskon(x):
    """Ubah ke pecahan 0-1. '18%' -> 0.18 ; 9 -> 0.09 ; 0.2 -> 0.2"""
    if pd.isna(x): return np.nan
    s = str(x).strip()
    persen = s.endswith('%')
    s = re.sub(r'[^\d.,\-]', '', s).replace(',', '.')
    try:
        v = float(s)
    except ValueError:
        return np.nan
    return round(v / 100 if (persen or v >= 1) else v, 4)

def parse_tanggal(x):
    if pd.isna(x): return pd.NaT
    s = str(x).strip()
    m = re.match(r'(\d{4})[-/](\d{1,2})[-/](\d{1,2})(?!\d)', s)   # 2023-04-22 / 2023/10/15 (YYYY-MM-DD)
    if m:
        try:
            return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))
        except ValueError:
            return pd.NaT
    if re.fullmatch(r'\d{1,2}/\d{1,2}/\d{4}', s):               # 21/09/2023 -> DD/MM/YYYY
        for f in ('%d/%m/%Y', '%m/%d/%Y'):
            t = pd.to_datetime(s, format=f, errors='coerce')
            if pd.notna(t): return t
    if re.fullmatch(r'\d{1,2}-\d{1,2}-\d{4}', s):               # 04-22-2023 -> MM-DD-YYYY
        for f in ('%m-%d-%Y', '%d-%m-%Y'):
            t = pd.to_datetime(s, format=f, errors='coerce')
            if pd.notna(t): return t
    return pd.to_datetime(s, dayfirst=True, errors='coerce')    # 03-Feb-2024, 08 November 2023

# ---------- Terapkan ----------
for key in ('rep', 'region', 'channel'):
    terap(key, title)
terap('kategori', std_kategori)
terap('bayar', std_bayar)
terap('cust', std_cust)
terap('diskon', std_diskon)

# Tanggal -> YYYY-MM-DD
if K['tanggal']:
    c = K['tanggal']
    df[c] = pd.to_datetime(df[c].apply(parse_tanggal)).dt.strftime('%Y-%m-%d')

# Angka uang -> float 2 desimal
for key in ('amount', 'cost', 'price'):
    c = K[key]
    if c:
        df[c] = pd.to_numeric(df[c].apply(ke_angka), errors='coerce').round(2)

# Angka bulat -> Int64
for key in ('id', 'qty'):
    c = K[key]
    if c:
        df[c] = pd.to_numeric(df[c].apply(ke_angka), errors='coerce').round().astype('Int64')

log.append('Standarisasi tanggal, teks, kategori, metode bayar, tipe customer, diskon, dan angka selesai')
display(df)

,product_id,sale_date,sales_rep,region,sales_amount,quantity_sold,product_category,unit_cost,unit_price,customer_type,discount,payment_method,sales_channel,region_and_sales_rep
0,1015,2023-09-21,David,South,4631.23,30,Food,261.56,371.40,Returning,0.20,Bank Transfer,Retail,SOUTH David
1,1002,2023-04-22,Eve,North,6551.23,9,Electronics,4398.16,4439.12,New,0.18,Wire Transfer,Online,north-Eve
2,1093,2023-04-21,Bob,West,4384.02,17,Furniture,3816.39,4209.44,Returning,0.11,<NA>,Retail,bob (WEST)
3,1075,2023-10-09,Charlie,West,8239.58,18,Clothing,2228.35,2682.34,New,0.13,Wire Transfer,Online,WEST Charlie
4,1003,2023-12-31,Alice,South,4775.59,30,Furniture,4190.28,4270.65,New,0.20,Cash,Online,SOUTH-Alice
5,1021,2023-02-11,Charlie,West,3761.15,32,Food,900.79,NaN,New,0.21,Cash,Online,CHARLIE (west)
6,1064,2023-01-04,Eve,East,7412.11,10,Electronics,4764.96,5074.42,New,0.12,Cash,Online,East-Eve
7,1088,2023-10-04,Bob,East,6116.75,40,Electronics,4904.93,5034.35,New,0.10,Credit Card,Retail,EAST-BOB
8,1088,2023-10-04,Bob,East,6116.75,40,Electronics,4904.93,5034.35,New,0.10,Credit Card,Retail,EAST-BOB
9,1030,2023-07-17,David,West,3023.48,19,Clothing,3049.33,3209.22,Returning,0.26,Cash,Online,WEST David


## 6. Duplikat & Nilai Kosong

In [6]:
# 6.1 Duplikat (dicek SETELAH standarisasi agar variasi penulisan tidak lolos)
if HAPUS_DUPLIKAT:
    n0 = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    log.append(f'Baris duplikat dihapus: {n0 - len(df)}')

# 6.2 Nilai kosong
kosong_sebelum = int(df.isna().sum().sum())

# Unit_Cost & Unit_Price -> median per kategori produk (cadangan: median keseluruhan)
for key in ('cost', 'price'):
    c = K[key]
    if c:
        if K['kategori']:
            df[c] = df[c].fillna(df.groupby(K['kategori'])[c].transform('median'))
        df[c] = df[c].fillna(df[c].median()).round(2)

# Sales_Amount -> median keseluruhan, Quantity -> median bulat
if K['amount']:
    df[K['amount']] = df[K['amount']].fillna(df[K['amount']].median()).round(2)
if K['qty']:
    df[K['qty']] = df[K['qty']].fillna(int(round(df[K['qty']].median())))

# Diskon kosong -> tanpa diskon
if K['diskon']:
    df[K['diskon']] = df[K['diskon']].fillna(ISI_DISKON_KOSONG)

# Kolom teks kategorikal -> 'Tidak Diketahui'
for key in ('rep', 'region', 'kategori', 'cust', 'bayar', 'channel'):
    c = K[key]
    if c:
        df[c] = df[c].fillna(ISI_TEKS_KOSONG)

log.append(f'Nilai kosong ditangani: {kosong_sebelum} -> {int(df.isna().sum().sum())} '
           f'(tersisa hanya pada kolom tanggal/ID bila memang tidak ada datanya)')
print('Sisa nilai kosong per kolom:')
print(df.isna().sum()[df.isna().sum() > 0].to_string() or 'Tidak ada')

Sisa nilai kosong per kolom:
Series([], )


## 7. Validasi Hasil

In [7]:
masalah = []

if K['diskon']:
    n = ((df[K['diskon']] < 0) | (df[K['diskon']] > 1)).sum()
    masalah.append(('Diskon di luar 0-1', int(n)))
if K['qty']:
    masalah.append(('Quantity <= 0', int((df[K['qty']] <= 0).sum())))
if K['cost'] and K['price']:
    masalah.append(('Unit_Price < Unit_Cost', int((df[K['price']] < df[K['cost']]).sum())))
if K['tanggal']:
    masalah.append(('Tanggal gagal dibaca (kosong)', int(df[K['tanggal']].isna().sum())))
if K['amount']:
    masalah.append(('Sales_Amount <= 0', int((df[K['amount']] <= 0).sum())))

print('=== HASIL VALIDASI ===')
for nama, n in masalah:
    print(f"{'✅' if n == 0 else '⚠️'} {nama}: {n}")

# Outlier (hanya dilaporkan, tidak dihapus)
for c in df.select_dtypes(include='number').columns:
    if c == K['id']:
        continue
    q1, q3 = df[c].quantile([.25, .75])
    iqr = q3 - q1
    out = df[(df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)]
    if len(out):
        print(f'ℹ️ Kolom "{c}": {len(out)} kemungkinan outlier (indeks {list(out.index)})')

=== HASIL VALIDASI ===
✅ Diskon di luar 0-1: 0
✅ Quantity <= 0: 0
⚠️ Unit_Price < Unit_Cost: 3
✅ Tanggal gagal dibaca (kosong): 0
✅ Sales_Amount <= 0: 0


## 7b. Enrichment
Menambah kolom turunan `profit_margin_pct` = (Unit_Price − Unit_Cost) / Unit_Price × 100.

In [8]:
df['profit_margin_pct'] = ((df[K['price']] - df[K['cost']]) / df[K['price']] * 100).round(2)

## 8. Laporan Sebelum vs Sesudah

In [9]:
print('=== RINGKASAN ===')
print(f'Baris   : {df_raw.shape[0]} -> {df.shape[0]}')
print(f'Kolom   : {df_raw.shape[1]} -> {df.shape[1]}')
print(f'Kosong  : {int(df_raw.isna().sum().sum())} -> {int(df.isna().sum().sum())}')

print('\n=== LOG PERUBAHAN ===')
for i, l in enumerate(log, 1):
    print(f'{i}. {l}')

print('\n=== NILAI UNIK KOLOM KATEGORI: SEBELUM vs SESUDAH ===')
for key in ('region', 'kategori', 'bayar', 'cust', 'channel', 'rep'):
    c = K[key]
    if not c: continue
    asal = [x for x in df_raw.columns if snake(x) == c]
    if asal:
        sebelum = sorted(df_raw[asal[0]].dropna().astype(str).str.strip().unique())
        sesudah = sorted(df[c].astype(str).unique())
        print(f'\n[{c}]  {len(sebelum)} variasi -> {len(sesudah)} nilai')
        print('  sebelum:', sebelum)
        print('  sesudah:', sesudah)

print('\n=== DATA SESUDAH CLEANSING ===')
display(df)
df.info()

=== RINGKASAN ===
Baris   : 34 -> 30
Kolom   : 16 -> 15
Kosong  : 12 -> 0

=== LOG PERUBAHAN ===
1. Nama kolom diseragamkan ke snake_case
2. Kolom redundan dihapus: ['category_copy', 'rep_name']
3. Spasi berlebih dirapikan & placeholder kosong diubah menjadi NaN
4. Baris kosong total dihapus: 0
5. Standarisasi tanggal, teks, kategori, metode bayar, tipe customer, diskon, dan angka selesai
6. Baris duplikat dihapus: 4
7. Nilai kosong ditangani: 10 -> 0 (tersisa hanya pada kolom tanggal/ID bila memang tidak ada datanya)

=== NILAI UNIK KOLOM KATEGORI: SEBELUM vs SESUDAH ===

[region]  12 variasi -> 4 nilai
  sebelum: ['EAST', 'East', 'NORTH', 'North', 'SOUTH', 'South', 'WEST', 'West', 'east', 'north', 'south', 'west']
  sesudah: ['East', 'North', 'South', 'West']

[product_category]  15 variasi -> 4 nilai
  sebelum: ['CLOTHING', 'Clothing', 'Cloths', 'Electronic', 'Electronic Goods', 'FOOD', 'FURNITURE', 'Food', 'Foods', 'Furnitue', 'Furniture', 'Furnitures', 'clothing', 'food', 'furnitu

,product_id,sale_date,sales_rep,region,sales_amount,quantity_sold,product_category,unit_cost,unit_price,customer_type,discount,payment_method,sales_channel,region_and_sales_rep,profit_margin_pct
0,1015,2023-09-21,David,South,4631.23,30,Food,261.56,371.40,Returning,0.20,Bank Transfer,Retail,SOUTH David,29.57
1,1002,2023-04-22,Eve,North,6551.23,9,Electronics,4398.16,4439.12,New,0.18,Wire Transfer,Online,north-Eve,0.92
2,1093,2023-04-21,Bob,West,4384.02,17,Furniture,3816.39,4209.44,Returning,0.11,Tidak Diketahui,Retail,bob (WEST),9.34
3,1075,2023-10-09,Charlie,West,8239.58,18,Clothing,2228.35,2682.34,New,0.13,Wire Transfer,Online,WEST Charlie,16.93
4,1003,2023-12-31,Alice,South,4775.59,30,Furniture,4190.28,4270.65,New,0.20,Cash,Online,SOUTH-Alice,1.88
5,1021,2023-02-11,Charlie,West,3761.15,32,Food,900.79,889.27,New,0.21,Cash,Online,CHARLIE (west),-1.30
6,1064,2023-01-04,Eve,East,7412.11,10,Electronics,4764.96,5074.42,New,0.12,Cash,Online,East-Eve,6.10
7,1088,2023-10-04,Bob,East,6116.75,40,Electronics,4904.93,5034.35,New,0.10,Credit Card,Retail,EAST-BOB,2.57
8,1030,2023-07-17,David,West,3023.48,19,Clothing,3049.33,3209.22,Returning,0.26,Cash,Online,WEST David,4.98
9,1100,2023-08-14,Bob,West,2198.74,43,Food,1100.81,1137.44,Returning,0.08,Bank Transfer,Online,West-bob,3.22


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   product_id            30 non-null     Int64  
 1   sale_date             30 non-null     object 
 2   sales_rep             30 non-null     object 
 3   region                30 non-null     object 
 4   sales_amount          30 non-null     float64
 5   quantity_sold         30 non-null     Int64  
 6   product_category      30 non-null     object 
 7   unit_cost             30 non-null     float64
 8   unit_price            30 non-null     float64
 9   customer_type         30 non-null     object 
 10  discount              30 non-null     float64
 11  payment_method        30 non-null     object 
 12  sales_channel         30 non-null     object 
 13  region_and_sales_rep  30 non-null     string 
 14  profit_margin_pct     30 non-null     float64
dtypes: Int64(2), float64(5), 

## 9. Simpan Dataset Sesudah Cleansing ke Google Drive

In [10]:
clean_path = f'{CLEAN_DIR}/{CLEAN_FILE}'
df.to_csv(clean_path, index=False, encoding='utf-8-sig')
df.to_excel(clean_path.replace('.csv', '.xlsx'), index=False)   # versi Excel (opsional)

print('✅ Dataset SEBELUM cleansing :', raw_path)
print('✅ Dataset SESUDAH cleansing :', clean_path)

✅ Dataset SEBELUM cleansing : /content/drive/MyDrive/Data_Cleansing/raw/sales_data_.xlsx
✅ Dataset SESUDAH cleansing : /content/drive/MyDrive/Data_Cleansing/clean/dataset_penjualan_bersih.csv
